# Notebook 3 — Missingness Mechanisms: MCAR, MAR, and MNAR

**Difficulty:** ⭐⭐⭐⭐ &nbsp;|&nbsp; **Estimated Time:** 2.5 hours &nbsp;|&nbsp; **Week 4 — Data Fundamentals & Preprocessing Pipelines**

---

## Why Does This Matter for ML?

Almost every real-world dataset has missing values. The naive approach — delete rows or fill with the mean — seems harmless, but **it can silently wreck your model**.

Here is why:

- If data is missing **because of its own value** (e.g., users with very low balances are less likely to report their balance), then filling those missing values with the average balance makes your model think "low balance" customers look normal — when they are actually the highest-risk group.
- In fraud detection this distinction is life-or-death for your model's accuracy. **A biased imputation strategy on MNAR data can make you miss 30–40% of actual fraud cases.**

The key insight of this notebook: **why the data is missing matters as much as what is missing.**

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Distinguish MCAR, MAR, and MNAR with real examples
2. Create synthetic datasets that simulate each mechanism
3. Run visual and statistical tests to diagnose missingness
4. Explain why MNAR cannot be detected from data alone
5. Know the correct strategy for each mechanism

## The Detective Analogy

Imagine you are a detective reviewing a filing cabinet full of credit card transaction folders. Some folders are missing a page. Before you decide how to handle the missing page, you need to ask: **WHY is the page missing?**

| Scenario | What happened | Technical name |
|----------|--------------|----------------|
| The office flooded and 10% of ALL folders lost a page — completely at random | The flood didn't care which folder it damaged | **MCAR** |
| Clerks in Branch A (a specific observed branch) routinely skipped filling in the credit-score page | The branch is known — it's in the data | **MAR** |
| Customers with very low balances refused to write down their balance | The missing value itself caused it to be missing | **MNAR** |

Each scenario demands a different response. Treating them all the same way introduces systematic errors.

## Section 1 — Setup: Build a Realistic Credit Card Dataset

We will create a synthetic dataset of 2,000 credit card transactions. The dataset will have **three columns** that we will artificially make missing, each using a different mechanism.

| Column | Missingness Mechanism | Reason |
|--------|-----------------------|--------|
| `device_type` | MCAR | Server glitch randomly lost records |
| `credit_score` | MAR | Missing more often for older age groups (systematic but observed) |
| `account_balance` | MNAR | Missing more often when balance is very low — the balance itself predicts its own missingness |

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency

# Reproducibility: any code that runs this notebook will get the same numbers
np.random.seed(42)

# Visual style — clean white background, slightly larger fonts for readability
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print('Libraries loaded successfully.')

In [ ]:
# ── Generate the base transaction dataset ────────────────────────────────────
N = 2000  # number of transactions

# --- Demographic features ---
# Age: uniform between 18 and 75 — used later to create MAR missingness in credit_score
age = np.random.randint(18, 76, size=N)

# Income: log-normal distribution — income is never negative and tends to be right-skewed
income = np.random.lognormal(mean=10.5, sigma=0.7, size=N).round(2)

# --- Transaction features ---
# Transaction amount: log-normal (most transactions are small, a few are large)
amount = np.random.lognormal(mean=4.5, sigma=1.2, size=N).round(2)

# Device type: categorical — 4 categories, mobile is most common in 2024
device_type_full = np.random.choice(
    ['mobile', 'desktop', 'tablet', 'POS'],
    size=N,
    p=[0.55, 0.25, 0.10, 0.10]  # probabilities must sum to 1
)

# Credit score: roughly normal, clipped between 300 (worst) and 850 (best)
credit_score_full = np.clip(
    np.random.normal(loc=650, scale=80, size=N),
    300, 850
).round(0).astype(int)

# Account balance: log-normal so most people have modest balances
# We add a deliberately heavy left tail: some people have near-zero balances
account_balance_full = np.clip(
    np.random.lognormal(mean=7.0, sigma=1.5, size=N),
    1, 500000
).round(2)

# Fraud label: ~3% base rate, higher if balance is low or credit score is low
fraud_prob = (
    0.03
    + 0.10 * (account_balance_full < 100)   # low balance → higher fraud risk
    + 0.05 * (credit_score_full < 550)       # poor credit → higher fraud risk
)
fraud_prob = np.clip(fraud_prob, 0, 1)        # probabilities must stay in [0,1]
is_fraud = np.random.binomial(1, fraud_prob)  # 1 = fraud, 0 = legitimate

# Country: used to illustrate MAR later (some countries have systematic data gaps)
country = np.random.choice(
    ['US', 'UK', 'DE', 'NG', 'IN'],
    size=N,
    p=[0.45, 0.20, 0.15, 0.10, 0.10]
)

# Assemble everything into a DataFrame — KEEP full values for now
df_full = pd.DataFrame({
    'transaction_id': range(1, N+1),
    'age': age,
    'income': income,
    'amount': amount,
    'device_type': device_type_full,
    'credit_score': credit_score_full,
    'account_balance': account_balance_full,
    'country': country,
    'is_fraud': is_fraud
})

print(f'Dataset shape: {df_full.shape}')
print(f'Fraud rate: {is_fraud.mean():.2%}')
df_full.head()

---
## Section 2 — MCAR: Missing Completely At Random

### Plain English First

**MCAR means the missingness is a pure accident.** Nothing about the data caused the value to be missing. A server crashed, a form got wet, someone accidentally deleted a column — the probability that any given row is missing is the same for every single row.

### Why This is the Easiest Case

Because the missing data is truly a random sample of all data, the rows with missing values look statistically identical to the rows without missing values. You have less data (bad) but no bias (manageable).

**Safe operations:** mean/median imputation, row deletion (listwise deletion)

### Formal Definition

Let $R$ be a binary indicator: $R=1$ if the value is missing. MCAR means:

$$P(R=1 \mid X_{obs}, X_{mis}) = P(R=1)$$

The probability of being missing does not depend on any data — observed or unobserved.

### Fraud Example

`device_type` is missing because a network switch failed during a 2-hour window on a Tuesday morning, randomly dropping 10% of all transaction records regardless of who made them.

In [ ]:
# ── Introduce MCAR missingness into device_type ───────────────────────────────
# Create a working copy so we can compare to df_full at any time
df = df_full.copy()

# MCAR: randomly select 10% of row indices — no condition on any column value
mcar_missing_indices = np.random.choice(
    df.index,
    size=int(0.10 * N),   # exactly 10% of rows
    replace=False          # each index chosen at most once
)
# Set device_type to NaN for those rows
df.loc[mcar_missing_indices, 'device_type'] = np.nan

print('=== MCAR: device_type ===')
print(f'Missing values: {df["device_type"].isna().sum()} ({df["device_type"].isna().mean():.1%})')
print()

# Key MCAR test: does the distribution of OTHER columns differ between missing and non-missing?
# If MCAR is true, both groups should look the same on `amount` and `age`
missing_mask = df['device_type'].isna()

print('--- MCAR diagnostic: comparing amount for missing vs non-missing device_type ---')
print(df.groupby(missing_mask)['amount'].describe().round(2))
print()
print('--- MCAR diagnostic: comparing age ---')
print(df.groupby(missing_mask)['age'].describe().round(2))
print()

# Two-sample t-test on `amount`: if p > 0.05 we cannot reject that the two groups have the same mean
# This is consistent with MCAR (though not proof)
t_stat, p_value = stats.ttest_ind(
    df.loc[missing_mask, 'amount'],
    df.loc[~missing_mask, 'amount']
)
print(f'T-test on amount (missing vs not missing device_type):')
print(f'  t = {t_stat:.3f},  p = {p_value:.3f}')
print(f'  Interpretation: p {'> 0.05 → no significant difference (consistent with MCAR)' if p_value > 0.05 else '< 0.05 → significant difference (NOT consistent with MCAR)'}')

In [ ]:
# ── Visual MCAR test: overlay distributions ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('MCAR Diagnostic — Comparing Distributions: Missing vs Non-Missing device_type',
             fontsize=13, fontweight='bold')

missing_mask = df['device_type'].isna()

# Left plot: transaction amount
# If distributions overlap heavily → missingness is unrelated to amount → consistent with MCAR
sns.kdeplot(
    df.loc[~missing_mask, 'amount'], ax=axes[0],
    label='device_type PRESENT', color='steelblue', linewidth=2
)
sns.kdeplot(
    df.loc[missing_mask, 'amount'], ax=axes[0],
    label='device_type MISSING', color='tomato', linewidth=2, linestyle='--'
)
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount ($)')
axes[0].legend()
# Add note about what to look for
axes[0].text(0.97, 0.95, 'MCAR: curves should overlap',
             transform=axes[0].transAxes, ha='right', va='top',
             fontsize=9, color='gray', style='italic')

# Right plot: age distribution
sns.kdeplot(
    df.loc[~missing_mask, 'age'], ax=axes[1],
    label='device_type PRESENT', color='steelblue', linewidth=2
)
sns.kdeplot(
    df.loc[missing_mask, 'age'], ax=axes[1],
    label='device_type MISSING', color='tomato', linewidth=2, linestyle='--'
)
axes[1].set_title('Age Distribution')
axes[1].set_xlabel('Age (years)')
axes[1].legend()
axes[1].text(0.97, 0.95, 'MCAR: curves should overlap',
             transform=axes[1].transAxes, ha='right', va='top',
             fontsize=9, color='gray', style='italic')

plt.tight_layout()
plt.show()
print('Observation: the two distributions overlap almost perfectly — the missing group'
      ' is indistinguishable from the full dataset. This is the MCAR signature.')

### Little's MCAR Test (Concept + Simplified Implementation)

**Little's MCAR test** (1988) is the standard formal statistical test for MCAR. It works by:

1. Grouping rows by their missingness **pattern** (which columns are missing)
2. For each pattern group, computing the mean of each observed variable
3. Comparing these group means to the overall mean via a chi-squared statistic
4. If means are similar across all groups → MCAR (p > 0.05 means we cannot reject MCAR)

The full implementation uses the `pyampute` or `rpy2` packages. Below is a simplified version using a chi-squared test on a single variable to illustrate the logic.

In [ ]:
# ── Simplified Little's MCAR test logic ───────────────────────────────────────
# The idea: bin the continuous variable (amount) into quartiles, then test
# whether the missingness rate is the same across all quartiles.
# Under MCAR: all quartiles should have ~10% missing → chi-squared test

# Create a temporary dataframe for this analysis
temp = df[['amount', 'device_type']].copy()
# pd.qcut divides into 4 equal-sized buckets based on quantile cutpoints
temp['amount_quartile'] = pd.qcut(temp['amount'], q=4, labels=['Q1','Q2','Q3','Q4'])
# Binary: 1 if device_type is missing
temp['device_missing'] = temp['device_type'].isna().astype(int)

# Cross-tabulation: rows = quartile, columns = missing (0 or 1)
contingency = pd.crosstab(temp['amount_quartile'], temp['device_missing'])
contingency.columns = ['Present', 'Missing']
# Add missingness rate column for readability
contingency['Missing Rate'] = (
    contingency['Missing'] / contingency.sum(axis=1)
).round(3)

print('=== Missingness rate of device_type by amount quartile ===')
print(contingency)
print()

# Chi-squared test on the 2-column contingency table (Present vs Missing counts)
chi2, p_chi2, dof, expected = chi2_contingency(contingency[['Present', 'Missing']])
print(f'Chi-squared statistic: {chi2:.3f}')
print(f'Degrees of freedom:    {dof}')
print(f'p-value:               {p_chi2:.3f}')
print()
if p_chi2 > 0.05:
    print('Result: p > 0.05 → Cannot reject MCAR. Missingness rates are uniform across quartiles.')
    print('        This is consistent with MCAR — the amount of a transaction does NOT predict')
    print('        whether device_type is missing.')
else:
    print('Result: p < 0.05 → Evidence AGAINST MCAR. Missingness rate varies by amount quartile.')

---
## Section 3 — MAR: Missing At Random

### Plain English First

**MAR does NOT mean "random" in everyday language.** Despite the name, it means there IS a pattern — but the pattern is explained by **other observed variables**.

Think of it this way: the missingness is systematic (not truly random), but the variable that explains the pattern is something **you can see in your data**.

### Formal Definition

$$P(R=1 \mid X_{obs}, X_{mis}) = P(R=1 \mid X_{obs})$$

Once you condition on (control for) the observed variables, the remaining missingness is random.

### Fraud Example

`credit_score` is missing more often for users **over age 60**. Older users are less likely to have been through a recent credit pull. The age is recorded (observed), so the pattern is explainable. The credit score value itself is not what's causing it to be missing — the age is.

In [ ]:
# ── Introduce MAR missingness into credit_score ──────────────────────────────
# Rule: older users are more likely to have credit_score missing
#   age < 40  → 5%  chance of missing (young users filled it in)
#   age 40-59 → 12% chance of missing
#   age >= 60 → 35% chance of missing (older users often didn't go through credit pull)

# Build a missingness probability for each row based on their age group
missing_prob_credit = np.where(
    df['age'] < 40,  0.05,
    np.where(
        df['age'] < 60, 0.12,
        0.35               # age >= 60
    )
)

# Sample a binary mask: 1 = missing, based on the age-dependent probability
mar_missing_mask = np.random.binomial(1, missing_prob_credit).astype(bool)
df.loc[mar_missing_mask, 'credit_score'] = np.nan

print('=== MAR: credit_score ===')
print(f'Missing values: {df["credit_score"].isna().sum()} ({df["credit_score"].isna().mean():.1%})')
print()

# Create age groups for the diagnostic bar chart
df['age_group'] = pd.cut(
    df['age'],
    bins=[17, 30, 45, 60, 100],
    labels=['18–30', '31–45', '46–60', '60+']
)

# Compute missingness rate for credit_score within each age group
mar_diagnostic = (
    df.groupby('age_group', observed=True)['credit_score']
    .apply(lambda x: x.isna().mean())
    .reset_index()
)
mar_diagnostic.columns = ['age_group', 'missing_rate']
print('--- MAR diagnostic: missing rate by age group ---')
print(mar_diagnostic.to_string(index=False))
print()
print('Pattern: missing rate INCREASES with age — this is the MAR signature.')
print('The missingness is NOT random overall, but it IS random once we know the age group.')

In [ ]:
# ── Visual MAR test: missing rate by age group ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('MAR Diagnostic — credit_score Missingness Explained by Age',
             fontsize=13, fontweight='bold')

# Left: bar chart of missing rate per age group
bars = axes[0].bar(
    mar_diagnostic['age_group'],
    mar_diagnostic['missing_rate'],
    color=['#4CAF50', '#FFC107', '#FF5722', '#F44336'],
    edgecolor='black', linewidth=0.5
)
# Add percentage labels on top of each bar
for bar, rate in zip(bars, mar_diagnostic['missing_rate']):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{rate:.1%}',
        ha='center', va='bottom', fontweight='bold'
    )
axes[0].set_title('credit_score Missing Rate by Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Proportion Missing')
axes[0].set_ylim(0, 0.45)
axes[0].axhline(df['credit_score'].isna().mean(), color='navy', linestyle='--',
                label=f'Overall avg: {df["credit_score"].isna().mean():.1%}')
axes[0].legend()

# Right: KDE comparison of credit_score distribution (present vs missing age groups)
# Under MAR: the actual DISTRIBUTION of credit scores present vs absent should look similar
# (because it's age driving missingness, not the credit score value itself)
cs_missing_mask = df['credit_score'].isna()
sns.kdeplot(
    df.loc[~cs_missing_mask, 'amount'], ax=axes[1],
    label='credit_score PRESENT', color='steelblue', linewidth=2
)
sns.kdeplot(
    df.loc[cs_missing_mask, 'amount'], ax=axes[1],
    label='credit_score MISSING', color='tomato', linewidth=2, linestyle='--'
)
axes[1].set_title('Transaction Amount: Does it Predict credit_score Missingness?')
axes[1].set_xlabel('Amount ($)')
axes[1].legend()
axes[1].text(0.97, 0.95,
             'MAR: other feature (age) predicts\nmissingness, not the value itself',
             transform=axes[1].transAxes, ha='right', va='top',
             fontsize=9, color='gray', style='italic')

plt.tight_layout()
plt.show()

---
## Section 4 — MNAR: Missing Not At Random

### Plain English First

**MNAR is the dangerous one.** The missingness is caused by the **value that is missing itself**. The data is hiding from you precisely because of what it would have shown.

### The Terrifying Implication

You **cannot detect MNAR from the data alone.** The reason is circular: to test whether missingness depends on the unobserved value, you would need to know the value — but the value is missing. You need external domain knowledge.

### Formal Definition

$$P(R=1 \mid X_{obs}, X_{mis}) \neq P(R=1 \mid X_{obs})$$

Even after conditioning on all observed variables, the missingness still depends on the unobserved value $X_{mis}$.

### Fraud Example

`account_balance` is missing more often when the account balance is very low (< \$100). This happens because:
- Accounts that have been drained (possible fraud) show no balance to report
- Customers in financial distress decline to share their balance
- The bank's API returns NULL for accounts with insufficient funds

**The consequence:** if you fill in missing balances with the mean balance (\$2,000+), you are telling your model that customers with missing balances look financially healthy — when they are actually the highest-risk group.

In [ ]:
# ── Introduce MNAR missingness into account_balance ───────────────────────────
# Rule: the lower the balance, the MORE likely it is to be missing
# This makes balance < $100 records almost always missing

# Compute a missingness probability that is INVERSELY related to the balance value
# We use a logistic-like function: sigmoid( -balance_scaled )
# Normalize balance to [0,1] range for the sigmoid input
balance_normalized = (df_full['account_balance'] - df_full['account_balance'].min()) / \
                     (df_full['account_balance'].max() - df_full['account_balance'].min())

# Sigmoid: 1 / (1 + exp(x)) where x grows with balance → P(missing) drops as balance rises
# Shift so that very low balances have ~60% chance of missing, high balances ~2%
mnar_missing_prob = 1 / (1 + np.exp(6 * balance_normalized - 0.5))
# The -0.5 shift ensures overall missingness is around 15–20%
mnar_missing_mask = np.random.binomial(1, mnar_missing_prob).astype(bool)
df.loc[mnar_missing_mask, 'account_balance'] = np.nan

print('=== MNAR: account_balance ===')
print(f'Missing values: {df["account_balance"].isna().sum()} ({df["account_balance"].isna().mean():.1%})')
print()

# MNAR smoking gun: compare fraud rates between missing vs non-missing balance
# If missingness is MNAR (missing because balance was low), then missing-balance rows
# should have HIGHER fraud rates (since low balance → higher fraud probability)
balance_missing_mask = df['account_balance'].isna()
fraud_rate_missing_balance     = df.loc[balance_missing_mask, 'is_fraud'].mean()
fraud_rate_nonmissing_balance  = df.loc[~balance_missing_mask, 'is_fraud'].mean()

print('=== MNAR Evidence: Fraud Rate by Balance Missingness ===')
print(f'Fraud rate when balance IS  missing: {fraud_rate_missing_balance:.1%}')
print(f'Fraud rate when balance NOT missing: {fraud_rate_nonmissing_balance:.1%}')
print()
print('INTERPRETATION:')
print('  The fraud rate is MUCH higher in the group with missing balance.')
print('  This is the MNAR smoking gun: the missingness IS carrying information about fraud.')
print('  If we delete these rows or fill with mean, we LOSE this signal.')

In [ ]:
# ── MNAR Visual: fraud rate and balance distribution ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('MNAR Evidence — Missing account_balance is a Fraud Signal',
             fontsize=13, fontweight='bold')

# Left: bar chart of fraud rates for missing vs non-missing balance
categories = ['Balance\nMISSING', 'Balance\nPRESENT']
fraud_rates = [fraud_rate_missing_balance, fraud_rate_nonmissing_balance]
colors = ['#D32F2F', '#388E3C']  # red for danger, green for safe

bars = axes[0].bar(categories, fraud_rates, color=colors, edgecolor='black',
                   linewidth=0.8, width=0.5)
for bar, rate in zip(bars, fraud_rates):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f'{rate:.1%}',
        ha='center', va='bottom', fontsize=13, fontweight='bold'
    )
axes[0].set_title('Fraud Rate by Balance Missingness')
axes[0].set_ylabel('Fraud Rate')
axes[0].set_ylim(0, max(fraud_rates) * 1.4)
axes[0].annotate(
    'MNAR: missing balance\n= strong fraud indicator!',
    xy=(0, fraud_rate_missing_balance),
    xytext=(0.6, fraud_rate_missing_balance + 0.05),
    arrowprops=dict(arrowstyle='->', color='black'),
    fontsize=9, color='darkred'
)

# Right: histogram of the TRUE balance values for rows that ended up missing
# This shows the MNAR mechanism: missing values cluster at low balances
# (In reality you can't see this plot — the values ARE missing — this is for illustration)
true_balance_of_missing = df_full.loc[mnar_missing_mask, 'account_balance']
true_balance_of_present = df_full.loc[~mnar_missing_mask, 'account_balance']

axes[1].hist(np.log1p(true_balance_of_present), bins=40, alpha=0.6,
             color='steelblue', label='Balance PRESENT', density=True)
axes[1].hist(np.log1p(true_balance_of_missing), bins=40, alpha=0.6,
             color='tomato', label='Balance MISSING (true values)', density=True)
axes[1].set_title('True Distribution of Missing vs Present Balance\n(log scale — red cluster at low values)')
axes[1].set_xlabel('log(1 + balance)')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].text(0.97, 0.97,
             'In practice you cannot see this!\nThis is the hidden MNAR pattern.',
             transform=axes[1].transAxes, ha='right', va='top',
             fontsize=8.5, color='gray', style='italic',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

plt.tight_layout()
plt.show()

### MNAR Strategy: Add a Binary Indicator Feature

Since you cannot recover the true value and cannot test for MNAR from the data, the best approach is:

1. **Create a `_was_missing` binary indicator feature** — this preserves the information that the value was missing (which is itself a signal)
2. **Then impute the missing value** (with mean, median, or a model) — so downstream algorithms can still use the column
3. **Use domain knowledge** — talk to the data source team to understand WHY values are missing

In [ ]:
# ── MNAR Strategy: create binary indicator feature ────────────────────────────
# Step 1: Before imputing, record WHICH rows had missing balance
# This is a new feature the ML model can use — missingness IS a signal
df['is_balance_missing'] = df['account_balance'].isna().astype(int)

print('=== New Feature: is_balance_missing ===')
print(df['is_balance_missing'].value_counts())
print()

# Step 2: Verify the indicator captures the fraud signal
# Group by the new indicator and compute fraud rate
fraud_by_indicator = (
    df.groupby('is_balance_missing')['is_fraud']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'fraud_rate', 'count': 'n_transactions'})
)
fraud_by_indicator.index = fraud_by_indicator.index.map({0: 'Balance Present', 1: 'Balance Missing'})
fraud_by_indicator['fraud_rate'] = fraud_by_indicator['fraud_rate'].map('{:.1%}'.format)
print('=== Fraud signal captured by is_balance_missing indicator ===')
print(fraud_by_indicator)
print()
print('The is_balance_missing = 1 group has a much higher fraud rate.')
print('This feature will help the ML model — do NOT lose this information!')
print()

# Step 3: Fill missing balance with median (after creating the indicator)
# We use median because balance is heavily right-skewed
balance_median = df['account_balance'].median()
df['account_balance_imputed'] = df['account_balance'].fillna(balance_median)
print(f'account_balance median used for imputation: ${balance_median:,.2f}')
print(f'Remaining NaN in account_balance_imputed: {df["account_balance_imputed"].isna().sum()}')

---
## Section 5 — Missingness Pattern Visualization

The `missingno` library creates beautiful missingness heatmaps — but it is not always available. Below we build the equivalent using seaborn.

The **missingness matrix** shows which rows are missing which columns. If two columns tend to be missing **at the same time**, that correlation tells you something about the data collection process.

In [ ]:
# ── Missingness pattern heatmap (missingno-style, built with seaborn) ─────────
# Columns of interest: the three we introduced missingness into
cols_with_missing = ['device_type', 'credit_score', 'account_balance']

# Build a binary matrix: 1 = missing, 0 = present
# We sample 200 rows to keep the plot readable
sample_idx = np.random.choice(df.index, size=200, replace=False)
miss_matrix = df.loc[sorted(sample_idx), cols_with_missing].isna().astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Missingness Pattern Analysis', fontsize=13, fontweight='bold')

# Left: nullity matrix — each row is a transaction, each column is a feature
# Black = missing, white = present
im = axes[0].imshow(
    miss_matrix.values.T,  # transpose so rows=features, cols=transactions
    aspect='auto',
    cmap='binary',
    interpolation='none'
)
axes[0].set_title('Nullity Matrix (200 transactions sample)\nBlack = Missing Value')
axes[0].set_yticks(range(len(cols_with_missing)))
axes[0].set_yticklabels(cols_with_missing)
axes[0].set_xlabel('Transaction index (sample)')
# No correlation in patterns → MCAR pattern tends to show no co-occurrence

# Right: missingness correlation heatmap
# High correlation between two columns' missingness = they tend to be missing together
# This can reveal structural data quality issues
full_miss_matrix = df[cols_with_missing].isna().astype(int)
miss_corr = full_miss_matrix.corr()

mask = np.zeros_like(miss_corr, dtype=bool)
np.fill_diagonal(mask, True)  # mask diagonal (always 1.0)
sns.heatmap(
    miss_corr,
    ax=axes[1],
    annot=True,
    fmt='.3f',
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    mask=mask,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Pearson correlation of missingness'}
)
axes[1].set_title('Missingness Correlation Matrix\n(Are missing values correlated across columns?)')

plt.tight_layout()
plt.show()
print('Note: low correlations between columns confirm that our three mechanisms are independent.')
print('In a real dataset, high correlations would suggest a shared structural cause (e.g., same API call fails for all fields).')

---
## Section 6 — Missingness Correlation with Other Features

A key diagnostic: create binary `_is_missing` indicators for each column, then correlate them with all other numeric features. High correlation reveals MAR patterns (other features predict missingness) and can hint at MNAR (but cannot prove it).

In [ ]:
# ── Missingness correlation matrix with observed features ─────────────────────
# Create binary indicators for all three missingness columns
df['device_type_missing']     = df['device_type'].isna().astype(int)
df['credit_score_missing']    = df['credit_score'].isna().astype(int)
df['account_balance_missing'] = df['account_balance'].isna().astype(int)

# Select numeric observed features + the three missingness indicators
numeric_cols = ['age', 'income', 'amount', 'is_fraud']
missing_indicators = ['device_type_missing', 'credit_score_missing', 'account_balance_missing']

# Correlation between missingness indicators and observed features
corr_data = df[numeric_cols + missing_indicators].corr()

# Extract only the cross-correlations we care about:
# rows = missingness indicators, columns = observed features
cross_corr = corr_data.loc[missing_indicators, numeric_cols]

print('=== Correlation of missingness indicators with observed features ===')
print(cross_corr.round(3))
print()
print('Reading guide:')
print('  device_type_missing  vs age:       ~0  → MCAR confirmed (age does NOT predict this missingness)')
print('  credit_score_missing vs age:        HIGH → MAR confirmed (age DOES predict this missingness)')
print('  account_balance_missing vs is_fraud: moderate → MNAR hint (missing balance correlates with fraud)')

# Visualize as a focused heatmap
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    cross_corr,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn',   # red = negative, green = positive correlation
    center=0,
    vmin=-0.6, vmax=0.6,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Pearson r'}
)
ax.set_title('Missingness Indicators vs Observed Features\n(High |r| → missingness is predictable → MAR or MNAR hint)',
             fontsize=11)
ax.set_yticklabels(
    ['device_type\n(MCAR)', 'credit_score\n(MAR)', 'account_balance\n(MNAR)'],
    rotation=0
)
plt.tight_layout()
plt.show()

---
## Section 7 — Summary Table

| | **MCAR** | **MAR** | **MNAR** |
|---|---|---|---|
| **What causes missingness?** | Pure chance — no pattern | Another observed variable | The missing value itself |
| **Example** | Server crash drops random records | Older users skip credit score | Low-balance users hide their balance |
| **Can you test for it?** | Yes — compare distributions across groups, Little's test | Yes — cross-tabulate with observed features | **No** — the evidence is hidden by definition |
| **Bias risk?** | Low — missing data is a random sample | Medium — ignoring MAR can bias estimates | **High** — deleting or naive imputation introduces systematic bias |
| **Safe strategy** | Mean/median/mode, row deletion | Model-based imputation (KNN, MICE) | Add `_was_missing` indicator, use domain knowledge |
| **Model consequence if ignored** | Reduced sample size only | Biased coefficient estimates | Systematically wrong predictions for the most important group |

---

## Section 8 — Final Dataset Summary

In [ ]:
# ── Final missingness summary report ─────────────────────────────────────────
print('=' * 65)
print('FINAL DATASET MISSINGNESS REPORT')
print('=' * 65)

# For each column, report: count missing, % missing, mechanism
mechanism_map = {
    'device_type':     'MCAR — random server outage',
    'credit_score':    'MAR  — driven by age group',
    'account_balance': 'MNAR — low balance users hide their balance'
}

for col in ['device_type', 'credit_score', 'account_balance']:
    n_missing = df[col].isna().sum()
    pct = df[col].isna().mean() * 100
    print(f'\n  Column: {col}')
    print(f'    Missing: {n_missing} / {N} ({pct:.1f}%)')
    print(f'    Mechanism: {mechanism_map[col]}')

print()
print('Key takeaways:')
print('  1. MCAR is safe to impute with simple statistics')
print('  2. MAR requires conditioning on the variable causing missingness')
print('  3. MNAR ALWAYS needs an indicator column before imputation')
print('  4. You cannot detect MNAR from the data alone — use domain knowledge')
print()
print('Next notebook: Imputation strategies (mean, KNN, iterative/MICE)')

---
## Self-Check Questions

Test your understanding before moving on. Try to answer in your own words first.

---

**Question 1:** In a medical study, patients with the worst outcomes are most likely to drop out before the study ends. What missingness mechanism is this? What are the implications for analyzing the remaining patients?

<details>
<summary>Click to reveal answer</summary>

**MNAR — Missing Not At Random.**

The missingness (dropout) is caused by the **very thing being measured** (severity of outcome). The sickest patients leave the study, so the remaining dataset is systematically biased toward healthier patients. 

Implication: any analysis on the remaining data will *underestimate* disease severity, *overestimate* treatment effectiveness, and *underestimate* mortality rates. This is called **survivor bias** and it is one of the most common and dangerous sources of bias in clinical research.

Strategy: use sensitivity analysis, add a "dropped out" indicator, use mixed models that handle missing outcomes, or report findings with explicit MNAR caveats.
</details>

---

**Question 2:** The `email` field is missing for users who signed up via the mobile app (all mobile users skipped that field during onboarding), but present for all web users. What missingness mechanism is this? How would you diagnose it?

<details>
<summary>Click to reveal answer</summary>

**MAR — Missing At Random.**

The missingness in `email` depends on `signup_channel` (mobile vs web) — an **observed** variable. Once you know the signup channel, the missingness is completely explained: all mobile users are missing email, all web users have email. The email value itself is not causing the missingness.

Diagnosis: cross-tabulate `email_is_missing` against `signup_channel`. You will see 100% missing for mobile, 0% for web — a clear MAR pattern.

Strategy: impute email using model-based methods conditioned on signup_channel and other features, OR simply keep `signup_channel` in the model since it fully explains the pattern.
</details>

---

**Question 3:** Should you always add a `_was_missing` indicator feature for MCAR columns? Why or why not?

<details>
<summary>Click to reveal answer</summary>

**Generally no — and adding one for MCAR can introduce noise.**

Under MCAR, the missing indicator is **uncorrelated with the target** (by definition). Adding it as a feature:
- Adds a column that contains no predictive information
- Adds noise to tree-based models (extra random splits with no gain)
- May spuriously correlate with the target by chance in small samples
- Increases model complexity with no benefit

The `_was_missing` indicator is valuable for **MNAR** (where missingness IS predictive of the target) and sometimes for **MAR** (where the pattern of missingness is meaningful). For MCAR: just impute and move on.

Exception: in a production system, you might still add a `_was_missing` indicator purely for **data quality monitoring** — not as a model feature, but to track when upstream data pipelines start failing.
</details>

---

**Question 4:** You can detect MAR from the data, but you cannot detect MNAR. Why is that? Explain the logical reason.

<details>
<summary>Click to reveal answer</summary>

**MAR** is detectable because the variable that predicts missingness (e.g., age) is **observed and present in your dataset**. You can literally look at it, group by it, and measure missingness rates across groups.

**MNAR** is fundamentally undetectable from the data alone because the variable that predicts missingness **is the missing value itself** — and that value is... missing. You cannot measure the relationship between a variable and its own probability of being missing if the values are absent.

It is a logical impossibility: "rows with low balance are more likely to have their balance missing" — but the rows with missing balance don't HAVE a balance for you to check.

This is why MNAR requires:
- **Domain knowledge** (talking to data engineers, understanding the data collection process)
- **External validation data** (if available)
- **Sensitivity analysis** (test how results change under different MNAR assumptions)

This is also why data scientists should always ask: *"Why is this value missing? Is there a process that would make high or low values more likely to be missing?"*
</details>